In [ ]:
# Vulnerability CLassifier (NN): i feel like I would use a normal ANN
# GOAL: Given vulnerability attributes and description predict a CVSS score (float value from 0-10)
# INPUT:
# OUTPUT: value between 0-10
# https://www.kaggle.com/code/cloudnineforreal/cvss-prediction

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

c:\Users\bride\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Upload/Understand Data

In [ ]:
cve_data = pd.read_csv("../data/cve.csv")
cve_data.head()

In [ ]:
# drop columns
drop_col = ["Unnamed: 0", "mod_date", "pub_date"]
cve_data.drop(columns=drop_col, inplace=True)

# fill na categorical columns as "UNKNOWN"
catgy_cols = ["access_authentication", "access_complexity", "access_vector", "impact_availability", "impact_confidentiality", "impact_integrity"]
cve_data[catgy_cols] = cve_data[catgy_cols].fillna("UNKNOWN")

# one hot encode categorical columns
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
catgy_encode = ohe.fit_transform(cve_data[catgy_cols])

# combine data
cve_data = pd.concat([cve_data, catgy_encode], axis=1)
cve_data.drop(columns=catgy_cols, inplace=True)

In [ ]:
cve_data["summary"]

In [ ]:
# vectorize summary field (SBERT)
model = SentenceTransformer("all-MiniLM-L6-v2") 
embeddings = model.encode(cve_data["summary"])
embeddings_df = pd.DataFrame(
    embeddings,
    columns=[f"SBERT_summary_{i}" for i in range(embeddings.shape[1])]
)

merged_cve_data = pd.concat([cve_data.drop(columns=["summary"]), embeddings_df], axis=1)

# tfidf_summary = TfidfVectorizer(max_features=500, stop_words="english")
# summary_feat = tfidf_summary.fit_transform(cve_data["summary"])
# # print(summary_feat[6])
# summary_feat_df = pd.DataFrame(
#     summary_feat.toarray(),
#     columns=[f"tfidf_summary_{i}" for i in range(summary_feat.shape[1])]
# )

# # combine data
# merged_cve_data = pd.concat([cve_data.drop(columns=["summary"]), summary_feat_df], axis=1)

# vectorize cve name field
tfidf_name = TfidfVectorizer(max_features=50, stop_words="english")
cwe_name_feat = tfidf_name.fit_transform(cve_data["cwe_name"])
name_feat_df = pd.DataFrame(
    cwe_name_feat.toarray(),
    columns=[f"tfidf_name_{i}" for i in range(cwe_name_feat.shape[1])]
)

# combine data
merged_cve_data = pd.concat([cve_data.drop(columns=["cwe_name"]), name_feat_df], axis=1)
merged_cve_data.head()

# Merged Data Run

In [2]:
merged_cve_data = pd.read_csv("C:\\Users\\bride\\OneDrive\\Desktop\\merged_cve.csv")
merged_cve_data.drop(columns="Unnamed: 0", inplace=True)

# split train/test
input_cols = merged_cve_data.loc[:, merged_cve_data.columns != "cvss"].columns
# input_cols
X = merged_cve_data[input_cols]
y = merged_cve_data["cvss"]

# drop object columns
X = X.select_dtypes(exclude="object")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## Model Definition

In [3]:
vuln_regr = XGBRegressor()
model = vuln_regr.fit(X_train, y_train)

In [4]:
# save model
vuln_regr.save_model("../model/xgb_regressor.json")

In [5]:
# model metrics
from sklearn.metrics import mean_squared_error, r2_score

# oob = rfr.oob_score_
# print(f"Out of Bag Score: {oob}")

predict = vuln_regr.predict(X_test)
mse = mean_squared_error(y_test, predict)
print(f"MSE: {mse}")

r2 = r2_score(y_test, predict)
print(f"R2 Value: {r2}")


MSE: 0.02625544087595387
R2 Value: 0.9933818153679375
